# WEB SCRAPING LAB – BOOKS TO SCRAPE
# Purpose: Scrape book data (UPC, Title, Price, Rating, Genre,
#          Availability, Description) using BeautifulSoup + pandas.
# Filters: min_rating, max_price

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [ ]:
# Rating conversion dictionary used by Books to Scrape
# THE WEBSITE STORES RSTINGS AS WORDS ("One", "Two", etc.)
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Base URL of the website
BASE_URL = "https://books.toscrape.com/"

# Main scraping function
# Filter by minimum rating and maximum price.
# Parameters: min_rating (int) and max_price(float)
def scrape_books(min_rating=4, max_price=20):
    all_books = []

    # Loop through all 50 pages of the website
    for page in range(1, 51):
        url = f"{BASE_URL}catalogue/page-{page}.html"
        response = requests.get(url)

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.select("article.product_pod")

        # loop through each book on the page
        for book in books:

            # Extract Title
            title = book.h3.a["title"]

            # Extract Rating (convert word to number)
            rating_class = book.p["class"][1]
            rating = RATING_MAP.get(rating_class, None)

            # Extract Price (clean encoding issues like Â£)
            # keep only digits + dot
            price_text = book.select_one("p.price_color").text
            price_clean = re.sub(r"[^\d.]", "", price_text)
            price = float(price_clean)

            # Apply Filter
            if rating < min_rating or price > max_price:
                continue

            # Detail page
            detail_url = BASE_URL + "catalogue/" + book.h3.a["href"]
            detail_resp = requests.get(detail_url)
            detail_soup = BeautifulSoup(detail_resp.text, "html.parser")

            # Extract UPC
            upc = detail_soup.select_one("table.table.table-striped tr:nth-of-type(1) td").text

            # Extract Availability
            availability = detail_soup.select_one("p.availability").text.strip()

            # Extract Description
            desc_tag = detail_soup.select_one("#product_description")
            description = desc_tag.find_next("p").text if desc_tag else "No description available"

            # Extract Genre (breadcrumb navigation)
            genre = detail_soup.select("ul.breadcrumb li")[2].text.strip()

            # Save book data
            all_books.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

    # convert list of dictionaries 
    df = pd.DataFrame(all_books)
    return df


# Run scraper
df_result = scrape_books(min_rating=4, max_price=20)
# Display first rows
print(df_result.head())
# Display total number of books scraped
print(f"\nTotal books scraped: {len(df_result)}")